# Buffered H3 Polygon Demo

This notebook demonstrates `get_buffered_h3_polygon` - a fast alternative to computing boundary children.

In [ ]:
from h3_boundary import cpp_geom_available
print(cpp_geom_available())  # True/False

In [ ]:
import time

import h3
import folium

from h3_boundary import (
    get_buffered_boundary_polygon_cpp,
    cell_boundary_from_children_cpp,
    get_buffered_h3_polygon,
    cell_boundary_to_geojson,
    cell_boundary_from_children,
    get_backend,
)

print(f"Backend: {get_backend()}")

## 1. Performance Comparison

In [ ]:
cell = h3.latlng_to_cell(37.7749, -122.4194, 2)
print(f"Cell: {cell} (Resolution {h3.get_resolution(cell)})")

# Buffered polygon (fast convex hull)
start = time.time()
buffered = get_buffered_boundary_polygon_cpp(cell, intermediate_res=7, use_convex_hull=True)
print(f"\nget_buffered_boundary_polygon (hull): {(time.time() - start)*1000:.2f} ms")
print(f"  Buffer: {buffered['properties']['buffer_meters']:.2f} meters")

# Boundary from children (slower, more accurate)
target_res = 9
start = time.time()
boundary = cell_boundary_from_children_cpp(cell, target_res)
print(f"\ncell_boundary_from_children(res={target_res}): {(time.time() - start)*1000:.2f} ms")
print(f"  Boundary cells: {boundary['properties']['num_boundary_cells']}")

## 2. Compare: Original vs Buffered vs Boundary from Children

In [ ]:
m = folium.Map(tiles='CartoDB positron')

# Boundary from children (blue - most accurate)
boundary_layer = folium.GeoJson(
    boundary,
    style_function=lambda x: {'color': 'blue', 'fillOpacity': 0.1, 'weight': 2},
    name='Boundary from Children (accurate)'
)
boundary_layer.add_to(m)

# Original cell (green dashed)
folium.GeoJson(
    cell_boundary_to_geojson(cell),
    style_function=lambda x: {'color': 'green', 'fillOpacity': 0, 'weight': 2, 'dashArray': '10,5'},
    name='Original Cell'
).add_to(m)

# Buffered polygon (red dashed)
folium.GeoJson(
    buffered,
    style_function=lambda x: {'color': 'red', 'fillOpacity': 0.05, 'weight': 2, 'dashArray': '5,5'},
    name='Buffered (fast approximation)'
).add_to(m)

folium.LayerControl().add_to(m)
# Fit the view to the data instead of a hardcoded zoom, so large
# boundaries (low-resolution cells) stay in view.
m.fit_bounds(boundary_layer.get_bounds())
m

## 3. Performance Table

Comparing `get_buffered_h3_polygon` (fast) vs `cell_boundary_from_children` (accurate)

In [ ]:
print("| Res | Buffered (ms) | Boundary from Children (ms) | Boundary Cells |")
print("|-----|---------------|-----------------------------|--------------------|")

for res in range(6, 12):
    cell = h3.latlng_to_cell(37.7749, -122.4194, res)
    target = min(res + 5, 15)  # Max target res 15
    
    start = time.time()
    buf = get_buffered_h3_polygon(cell)
    buf_time = (time.time() - start) * 1000
    
    start = time.time()
    bound = cell_boundary_from_children(cell, target)
    bound_time = (time.time() - start) * 1000
    
    n_cells = bound['properties']['num_boundary_cells']
    print(f"| {res:3} | {buf_time:13.2f} | {bound_time:27.2f} | {n_cells:18} |")